# bioseq bench `report.json` viewer

Self-contained notebook for inspecting a `report.json` produced by
`./build.sh bench`. The bench writes the file to `build/<preset>/report.json`
by default — the path below points at the production `bench-opt` preset.

## What this notebook shows (top to bottom)

1. **Headline verdict** — top-3 fastest bioseq variants vs minimap2 across
   sequence lengths. The single most important answer to "did we ship".
2. **Stage-by-stage breakdown** — where the wall-clock goes, for the same
   top-3 bioseq variants vs minimap2.
3. **Speedup vs scalar baseline** — every bioseq variant compared head-to-
   head with `bioseq KSW2 + scalar-DP + scalar-mm`. Positive = faster.
4. **Full algorithm grid wall-clock** — every (chaining, minimizer)
   combination across sequence lengths. Use to confirm the headline
   ranking holds for variants the top-3 chart hides.
5. **All chart-builder outputs grouped by tier** — k/w sweep, latency
   distributions, accuracy / quality diagnostics, PMU counters. Read
   when a deep-dive is needed.

The data comes from the bench's chart_builder which writes a stable
`bioseq-bench-report-v1` JSON schema. Every chart includes its own short
and long descriptions inside the JSON so the rendering loop in the last
section is generic.


In [1]:
# ----- configuration -----
# Override REPORT_PATH if the bench wrote elsewhere (e.g. the long-only
# sweep emits `report-long.json`).

REPORT_PATH = '../../../build/bench-opt/report.json'

import json
import math
import os
import sys

import plotly.graph_objects as go
import plotly.io
from IPython.display import display, Markdown

with open(REPORT_PATH, 'r') as report_file:
    report = json.load(report_file)

charts = report.get('charts', {})
print(f'Loaded {os.path.abspath(REPORT_PATH)}')
print(f'  schema:           {report.get("schema", "<missing>")}')
print(f'  sample_count:     {report.get("sample_count", "?")}')
print(f'  algorithm_count:  {report.get("algorithm_count", "?")}')
print(f'  chart_count:      {len(charts)}')

# Production label naming. The cleanup round consolidated the
# minimizer matrix down to three rows: scalar (the reference),
# SIMD-minimap2-slow (the legacy SIMD path kept for comparison) and
# two-stack-nthash (production). Anything else in `report.json` either
# came from a previous build or is a different chaining backbone.

MINIMAP2_REF      = 'minimap2 (DP chaining)'
SCALAR_BASELINE   = 'bioseq KSW2 + scalar-DP + scalar-mm'
SIMD_LEGACY       = 'bioseq KSW2 + SIMD-DP + SIMD-minimap2-slow-mm'
TWO_STACK_NTHASH  = 'bioseq KSW2 + SIMD-DP + two-stack-nthash-mm'


def shorten(label: str) -> str:
    '''Trim verbose labels so chart legends stay readable.'''
    if label == MINIMAP2_REF:
        return 'minimap2 (DP)'
    if label.startswith('bioseq '):
        return label[len('bioseq '):]
    return label


def is_main_grid_label(label: str) -> bool:
    '''Filter out the k/w-sweep and the rmq-repro regression rows.

    These are orthogonal sweeps that produce many variants of the same
    base algorithm; they make the headline comparison unreadable.
    '''
    if '(k=' in label or '(w=' in label:
        return False
    if '(rmq-repro)' in label:
        return False
    return True


def length_bucket_sort_key(label: str) -> float:
    '''Best-effort numeric sort for "short", "medium", "long", ... labels.'''
    canonical = {
        'short (<1kb)':       300,
        'medium (1k-10k)':    5_000,
        'long (10k-1M)':      50_000,
        'very long (1M-3M)':  1_000_000,
        'xl (5M-15M)':        5_000_000,
    }
    return canonical.get(label, 0.0)

Loaded /Users/styczynski/Code/bioseqdb/main/build/bench-opt/report.json
  schema:           bioseq-bench-report-v1
  sample_count:     3173
  algorithm_count:  43
  chart_count:      36


## 1. Headline verdict — top-3 bioseq variants vs `minimap2`

> **What this measures.** Mean wall-clock per (query, target) alignment
> pair, in milliseconds, plotted against the query length bucket. Each
> point is the average across the bench's per-pair sample replicates.
> Lower bars are better — these are end-to-end pipeline times that
> include minimizer extraction, chaining, base-level alignment, and
> MAPQ assignment.
>
> **What we're filtering for.** Among the bioseq variants the bench ran,
> only the **three fastest** at each length bucket are plotted, plus the
> `minimap2 (DP chaining)` reference. This keeps the chart readable
> while making it impossible to cherry-pick a slow row to look good
> against. The production minimizer (`two-stack-nthash`) should appear
> in every length bucket's top-3 if the V12-era engineering is intact.


In [2]:
def speed_chart_to_rows(chart, *, only_main_grid=True):
    '''Return [(label, [(x, y), ...])] from a plotly chart.

    By default skips k/w-sweep and rmq-repro rows that confuse the
    headline comparison. Pass `only_main_grid=False` to keep everything.
    '''
    rows = []
    for trace in chart.get('plotly', {}).get('data', []):
        label = trace.get('name', '')
        if only_main_grid and not is_main_grid_label(label):
            continue
        xs    = trace.get('x', [])
        ys    = trace.get('y', [])
        rows.append((label, list(zip(xs, ys))))
    return rows


def pick_top_n_overall(rows, n=3):
    '''Pick the n bioseq variants with the lowest *normalised* total
    wall-clock across length buckets.

    Normalisation: each bucket contributes the variant's mean ms divided
    by the SLOWEST variant's mean ms in the same bucket (so a bucket
    where every variant takes ~1 s doesn't drown a bucket where every
    variant takes ~10 ms). The resulting per-bucket ratios are summed
    across buckets; the n bioseq variants with the lowest sum win.

    Variants that are missing data in a bucket are penalised (treated as
    1.0 — i.e. as slow as the slowest in that bucket) so a variant that
    only ran on small inputs can't sneak into the headline by coincidence.
    '''
    bioseq_rows = [(lbl, dict(pts)) for lbl, pts in rows
                   if lbl != MINIMAP2_REF and lbl.startswith('bioseq ')]
    if not bioseq_rows:
        return set()
    all_xs = sorted({x for _lbl, x_to_y in bioseq_rows for x in x_to_y})
    bucket_max = {}
    for x in all_xs:
        bucket_max[x] = max((x_to_y.get(x, 0.0) or 0.0)
                            for _lbl, x_to_y in bioseq_rows)
        if bucket_max[x] == 0:
            bucket_max[x] = 1.0
    scores = []
    for lbl, x_to_y in bioseq_rows:
        score = 0.0
        for x in all_xs:
            y = x_to_y.get(x)
            score += (y / bucket_max[x]) if (y is not None and y > 0) else 1.0
        scores.append((score, lbl))
    scores.sort()
    return {lbl for _s, lbl in scores[:n]}


speed_chart = charts.get('seq_length_vs_speed_mean')
if speed_chart is None:
    display(Markdown('**`seq_length_vs_speed_mean` is missing — re-run the bench.**'))
else:
    rows = speed_chart_to_rows(speed_chart)
    chosen = pick_top_n_overall(rows, n=3)
    fig = go.Figure()
    palette_iter = iter(['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd', '#8c564b',
                         '#e377c2', '#bcbd22', '#17becf'])
    minimap2_rows = [(lbl, pts) for lbl, pts in rows if lbl == MINIMAP2_REF]
    for lbl, pts in minimap2_rows:
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='lines+markers',
            name=shorten(lbl),
            line=dict(color='#000000', width=3, dash='dot'),
            marker=dict(symbol='diamond', size=10),
        ))
    for lbl, pts in rows:
        if lbl not in chosen:
            continue
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        colour = next(palette_iter, '#666666')
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='lines+markers',
            name=shorten(lbl),
            line=dict(color=colour, width=2),
            marker=dict(size=8),
        ))
    fig.update_layout(
        title='Top-3 bioseq variants per length-bucket vs minimap2 (mean wall-clock)',
        xaxis=dict(title='Query length (bp, log scale)', type='log'),
        yaxis=dict(title='Mean wall-clock per pair (ms, log scale)', type='log'),
        legend_title='Variant (top-3 bioseq + minimap2 reference)',
        height=520,
    )
    fig.show()
    display(Markdown(
        f'> **{len(chosen)} bioseq variants survived the top-3-per-bucket filter.** '
        f'minimap2 reference is dotted black; bioseq variants are solid coloured. '
        f'Production minimizer is `{shorten(TWO_STACK_NTHASH)}`.'))

> **3 bioseq variants survived the top-3-per-bucket filter.** minimap2 reference is dotted black; bioseq variants are solid coloured. Production minimizer is `KSW2 + SIMD-DP + two-stack-nthash-mm`.

## 2. Where does the wall-clock go? (per-stage breakdown)

> **What this measures.** Per-pipeline-phase mean wall-clock per call,
> stacked into a bar per algorithm. Phases are the instrumentation
> regions registered in `minimap_port.cppm` and `chaining_*.cppm`:
> - `mm_sketch` / `mm_sketch_simd`: minimizer extraction.
> - `target_index_build` / `collect_seed_hits`: anchor production.
> - `mg_lchain_dp` / `mg_lchain_simd` / `mg_lchain_rmq`: chaining DP.
> - `mm_est_err`: divergence estimate from chain anchors.
> - `mm_align_skeleton`: KSW2 base-level alignment (the giant block at
>   the top of every bar — usually >50 % of total wall-clock).
>
> The chart filters to the same top-3 bioseq variants surfaced above,
> plus minimap2. Use it to confirm where the production minimizer
> actually saves work versus the scalar baseline.


In [3]:
DOWNSTREAM_PHASE_ORDER = (
    'mm_sketch',
    'mm_sketch_simd',
    'target_index_build',
    'collect_seed_hits',
    'mg_lchain_dp',
    'mg_lchain_simd',
    'mg_lchain_rmq',
    'mm_est_err',
    'mm_gen_regs',
    'mm_select_sub',
    'mm_set_mapq',
    'mm_set_parent',
    'mm_align_skeleton',
)


def render_phase_breakdown(chart_id, length_human):
    chart = charts.get(chart_id)
    if chart is None:
        return False
    # Each trace is one phase; trace.name is phase name, x is config
    # label, y is per-pair phase ms.
    phases_by_name = {}
    config_set = []
    for trace in chart.get('plotly', {}).get('data', []):
        phase = trace.get('name')
        for cfg, value in zip(trace.get('x', []), trace.get('y', [])):
            if not is_main_grid_label(cfg):
                continue
            phases_by_name.setdefault(phase, {})[cfg] = value
            if cfg not in config_set:
                config_set.append(cfg)
    if not config_set:
        return False

    # Determine the top-3 bioseq variants for this bucket using total
    # wall-clock — recompute locally to keep the chart self-contained.
    def total_for(cfg):
        return sum(phases_by_name.get(p, {}).get(cfg, 0.0) or 0.0
                   for p in phases_by_name)
    bioseq_configs = [c for c in config_set
                      if c != MINIMAP2_REF and c.startswith('bioseq ')]
    bioseq_configs.sort(key=total_for)
    top3 = bioseq_configs[:3]
    keep = ([MINIMAP2_REF] if MINIMAP2_REF in config_set else []) + top3

    fig = go.Figure()
    for phase in DOWNSTREAM_PHASE_ORDER:
        if phase not in phases_by_name:
            continue
        ys = [phases_by_name[phase].get(c, 0.0) or 0.0 for c in keep]
        if all(y == 0 for y in ys):
            continue
        fig.add_trace(go.Bar(name=phase, x=[shorten(c) for c in keep], y=ys))
    fig.update_layout(
        barmode='stack',
        title=f'Per-stage wall-clock — {length_human} (top-3 bioseq + minimap2)',
        xaxis_title='Variant',
        yaxis_title='Mean ms / call',
        height=480,
    )
    fig.show()
    display(Markdown(
        f'> Bars show {len(keep)} configurations: minimap2 (when present) and '
        f'the top-{min(3, len(top3))} fastest bioseq variants on this bucket. '
        f'Stack order matches the pipeline phase order in `minimap_port.cppm`.'))
    return True


for chart_id, human in [
    ('phase_breakdown_long',      'long sequences (10 k–1 M bp)'),
    ('phase_breakdown_very_long', 'very long sequences (1–3 M bp)'),
]:
    rendered = render_phase_breakdown(chart_id, human)
    if not rendered:
        display(Markdown(f'_`{chart_id}` not present in the report._'))

> Bars show 3 configurations: minimap2 (when present) and the top-3 fastest bioseq variants on this bucket. Stack order matches the pipeline phase order in `minimap_port.cppm`.

> Bars show 3 configurations: minimap2 (when present) and the top-3 fastest bioseq variants on this bucket. Stack order matches the pipeline phase order in `minimap_port.cppm`.

## 3. Speedup table — every variant vs the scalar baseline

> **What this measures.** Per-bucket relative speedup of each bioseq
> variant compared to `bioseq KSW2 + scalar-DP + scalar-mm`.
>
> Cell = `(scalar_ms − variant_ms) / scalar_ms × 100 %`. Positive
> numbers mean *faster than scalar*. The historical engineering goal was
> ≥10 % faster than scalar at the >1 M bp bucket — see
> `RESEARCH/ENGINEERING_REPORT_V2.md` for context.


In [4]:
# Length-bucket ranges for the speedup table. Boundaries are wider than
# the matrix_runner generator targets (e.g. xl pairs are ~4.96 Mbp not
# 5 Mbp exactly) so a noisy fragment length still lands in the right
# column.
LENGTH_BUCKETS = [
    ('< 1 kbp (short)',         0,             1_000),
    ('1 k – 10 k (medium)',     1_000,         10_000),
    ('10 k – 500 k (long)',     10_000,        500_000),
    ('500 k – 2 M (very long)', 500_000,       2_000_000),
    ('2 M – 15 M (xl)',         2_000_000,     15_000_000),
]


def bucket_index(x):
    for i, (_name, lo, hi) in enumerate(LENGTH_BUCKETS):
        if lo <= x < hi:
            return i
    return None


def aggregate_to_buckets(rows):
    '''Return {label: [mean_per_bucket_or_None, ...]} aggregating per bucket.'''
    out = {}
    for lbl, pts in rows:
        sums = [0.0] * len(LENGTH_BUCKETS)
        counts = [0] * len(LENGTH_BUCKETS)
        for x, y in pts:
            i = bucket_index(x)
            if i is None or y is None:
                continue
            sums[i] += y
            counts[i] += 1
        out[lbl] = [
            (sums[i] / counts[i]) if counts[i] > 0 else None
            for i in range(len(LENGTH_BUCKETS))
        ]
    return out


speed_chart = charts.get('seq_length_vs_speed_mean')
if speed_chart is None:
    display(Markdown('**`seq_length_vs_speed_mean` missing — re-run the bench.**'))
else:
    rows = speed_chart_to_rows(speed_chart)  # main grid only — skips k/w sweep
    bucket_means = aggregate_to_buckets(rows)
    scalar_means = bucket_means.get(SCALAR_BASELINE)
    if scalar_means is None:
        display(Markdown(
            f'**Scalar reference `{SCALAR_BASELINE}` not in the chart — '
            'cannot compute relative speedups.**'))
    else:
        # Column header per length bucket (only those with scalar data).
        active_buckets = [i for i, m in enumerate(scalar_means) if m is not None]
        header = ['Variant'] + [LENGTH_BUCKETS[i][0] for i in active_buckets]
        lines = ['| ' + ' | '.join(header) + ' |',
                 '|' + '|'.join(['---:'] * len(header)) + '|']
        # Sort: minimap2 first, then bioseq variants by best (largest)
        # average speedup across active buckets so the fastest rows
        # surface at the top.
        def avg_speedup(lbl):
            means = bucket_means.get(lbl, [])
            deltas = []
            for i in active_buckets:
                ref = scalar_means[i]
                cur = means[i] if i < len(means) else None
                if ref in (None, 0) or cur is None:
                    continue
                deltas.append((ref - cur) / ref * 100.0)
            if not deltas:
                return -math.inf
            return sum(deltas) / len(deltas)
        ordered = sorted((lbl for lbl in bucket_means if lbl != SCALAR_BASELINE),
                         key=lambda l: -avg_speedup(l))
        for lbl in ordered:
            cells = [shorten(lbl)]
            means = bucket_means[lbl]
            for i in active_buckets:
                ref = scalar_means[i]
                cur = means[i]
                if ref in (None, 0) or cur is None:
                    cells.append('—')
                else:
                    delta = (ref - cur) / ref * 100.0
                    cells.append(f'{delta:+.1f} %')
            lines.append('| ' + ' | '.join(cells) + ' |')
        display(Markdown('\n'.join(lines)))

| Variant | < 1 kbp (short) | 1 k – 10 k (medium) | 10 k – 500 k (long) | 500 k – 2 M (very long) | 2 M – 15 M (xl) |
|---:|---:|---:|---:|---:|---:|
| KSW2 + SIMD-DP + two-stack-nthash-mm | +0.9 % | +7.9 % | +10.8 % | +7.8 % | +4.7 % |
| KSW2 + SIMD-DP + scalar-mm | -0.9 % | +0.3 % | +2.5 % | +2.3 % | — |
| minimap2 (DP) | +46.3 % | -28.1 % | -28.0 % | -0.7 % | +15.4 % |
| Block + SIMD-DP + two-stack-nthash-mm | -1.0 % | +2.9 % | +2.8 % | -4.9 % | — |
| KSW2 + SIMD-DP + SIMD-minimap2-slow-mm | -1.4 % | -2.6 % | -1.0 % | +1.0 % | -0.8 % |
| KSW2 + RMQ + scalar-mm | -0.6 % | -2.9 % | -2.8 % | -4.7 % | — |
| minimap2 (RMQ chaining) | +46.4 % | -30.3 % | -27.8 % | -2.1 % | — |
| Block + SIMD-DP + scalar-mm | -3.0 % | -4.6 % | -4.5 % | -9.3 % | — |
| Block + scalar-DP + scalar-mm | -2.3 % | -6.3 % | -6.4 % | -11.3 % | — |
| Block + SIMD-DP + SIMD-minimap2-slow-mm | -3.3 % | -7.9 % | -10.8 % | -11.2 % | — |
| Block + RMQ + scalar-mm | -1.9 % | -6.2 % | -9.2 % | -16.9 % | — |

## 4. Full algorithm grid — wall-clock per length bucket

> **What this measures.** Same wall-clock data as section 1 but with
> *all* bioseq variants visible, not just the top-3. Use to verify the
> top-3 ranking holds across length buckets and to spot variants that
> are competitive in one bucket but slow in another.
>
> The chart pivots on length so a slope change between two buckets
> indicates the variant scales differently from the scalar baseline.


In [5]:
speed_chart = charts.get('seq_length_vs_speed_mean')
if speed_chart is None:
    display(Markdown('**`seq_length_vs_speed_mean` missing — re-run the bench.**'))
else:
    rows = speed_chart_to_rows(speed_chart)  # main grid only
    fig = go.Figure()
    palette_iter = iter([
        '#000000',  # minimap2
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
    ])
    for lbl, pts in rows:
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        is_minimap2 = (lbl == MINIMAP2_REF)
        is_production = (lbl == TWO_STACK_NTHASH)
        colour = next(palette_iter, '#aaaaaa')
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='lines+markers',
            name=shorten(lbl),
            line=dict(
                color=colour,
                width=3 if (is_minimap2 or is_production) else 1.5,
                dash='dot' if is_minimap2 else 'solid',
            ),
            marker=dict(size=8 if (is_minimap2 or is_production) else 5),
        ))
    fig.update_layout(
        title='Every main-grid variant — mean wall-clock per pair (k/w sweep excluded)',
        xaxis=dict(title='Query length (bp, log scale)', type='log'),
        yaxis=dict(title='Mean wall-clock per pair (ms, log scale)', type='log'),
        legend_title='Configuration (production = thick solid; minimap2 = dotted)',
        height=600,
    )
    fig.show()

## 5. Full per-stage breakdown — every variant

> Same stacked bars as section 2 but for every variant in the bench.
> Useful for diagnosing which stage explains an unexpected total
> wall-clock — e.g. when a SIMD-DP chain wins on chaining time but loses
> in `collect_seed_hits` because of minimizer-set distribution drift.


In [6]:
def render_full_phase_breakdown(chart_id, length_human):
    chart = charts.get(chart_id)
    if chart is None:
        return False
    phases_by_name = {}
    config_set = []
    for trace in chart.get('plotly', {}).get('data', []):
        phase = trace.get('name')
        for cfg, value in zip(trace.get('x', []), trace.get('y', [])):
            if not is_main_grid_label(cfg):
                continue
            phases_by_name.setdefault(phase, {})[cfg] = value
            if cfg not in config_set:
                config_set.append(cfg)
    if not config_set:
        return False
    fig = go.Figure()
    for phase in DOWNSTREAM_PHASE_ORDER:
        if phase not in phases_by_name:
            continue
        ys = [phases_by_name[phase].get(c, 0.0) or 0.0 for c in config_set]
        if all(y == 0 for y in ys):
            continue
        fig.add_trace(go.Bar(name=phase, x=[shorten(c) for c in config_set], y=ys))
    fig.update_layout(
        barmode='stack',
        title=f'Per-stage wall-clock — {length_human} (every variant)',
        xaxis_title='Variant',
        yaxis_title='Mean ms / call',
        height=520,
        xaxis=dict(tickangle=-30),
    )
    fig.show()
    return True


for chart_id, human in [
    ('phase_breakdown_long',      'long sequences (10 k–1 M bp)'),
    ('phase_breakdown_very_long', 'very long sequences (1–3 M bp)'),
    ('phase_breakdown_medium',    'medium sequences (1 k–10 k bp)'),
    ('phase_breakdown_short',     'short sequences (<1 k bp)'),
]:
    rendered = render_full_phase_breakdown(chart_id, human)
    if not rendered:
        display(Markdown(f'_`{chart_id}` not present in the report._'))

## 6. Auto-rendered chart-builder outputs (tiered)

> Everything else the chart_builder writes, grouped by tier so the
> reader can drill into a specific concern (k/w sweep, latency
> distribution, accuracy heatmap, PMU counters, …) without scrolling
> through unrelated charts. Each chart includes its own description
> rendered after the figure.


In [7]:
TIER_SPECS = [
    ('Tier 0 — Largest-bucket head-to-head (bioseq vs minimap2)',
     lambda cid: cid.startswith('headline_largest_')),
    ('Tier 1 — Headline summary',
     lambda cid: cid.startswith('headline_') and not cid.startswith('headline_largest_')),
    ('Tier 2 — Minimizer (k, w) parameter sweep',
     lambda cid: cid.startswith('kmer_window_')),
    ('Tier 3 — Latency distributions and tail behaviour',
     lambda cid: cid in {
         'seq_length_vs_speed_p99',
         'latency_distribution_box',
         'p99_over_mean_tail_amplification',
         'latency_cdf_per_algorithm',
         'seq_length_vs_memory',
     }),
    ('Tier 4 — Quality diagnostics',
     lambda cid: cid in {
         'position_offset_histogram',
         'mapq_distribution',
         'mapq_calibration',
         'algorithm_summary_table',
         'seq_length_vs_accuracy',
         'error_rate_vs_accuracy',
         'accuracy_heatmap_algorithm_vs_length',
         'accuracy_heatmap_algorithm_vs_length_error',
         'pareto_accuracy_vs_latency',
     }),
]


# Sections 1, 2, 3, 4, 5 above already render the most useful charts
# (seq_length_vs_speed_mean, phase_breakdown_*) — exclude them here so
# the rendering loop doesn't repeat them.
ALREADY_RENDERED = {
    'seq_length_vs_speed_mean',
    'phase_breakdown_short',
    'phase_breakdown_medium',
    'phase_breakdown_long',
    'phase_breakdown_very_long',
}


assigned = set()
for tier_label, predicate in TIER_SPECS:
    matching = sorted(cid for cid in charts
                      if cid not in assigned
                      and cid not in ALREADY_RENDERED
                      and predicate(cid))
    if not matching:
        continue
    display(Markdown(f'### {tier_label}'))
    for cid in matching:
        chart = charts[cid]
        short = chart.get('short_description', cid)
        long  = chart.get('long_description', '')
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'#### `{cid}` — _no plotly payload, skipped_'))
            assigned.add(cid)
            continue
        display(Markdown(f'#### `{cid}` — {short}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
            assigned.add(cid)
            continue
        figure.show()
        if long:
            display(Markdown(f'> {long}'))
        assigned.add(cid)


# Anything that didn't match a tier (e.g. user-added chart names) lands
# in a final catch-all so the notebook never silently drops a chart.
leftover = sorted(cid for cid in charts
                  if cid not in assigned and cid not in ALREADY_RENDERED)
if leftover:
    display(Markdown('### Other charts (uncategorised)'))
    for cid in leftover:
        chart = charts[cid]
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'#### `{cid}` — _no plotly payload, skipped_'))
            continue
        display(Markdown(
            f'#### `{cid}` — {chart.get("short_description", cid)}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
            figure.show()
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
        long = chart.get('long_description', '')
        if long:
            display(Markdown(f'> {long}'))

### Tier 0 — Largest-bucket head-to-head (bioseq vs minimap2)

#### `headline_largest_per_stage_branch_miss_rate` — Per-stage branch miss rate (every algorithm) — largest sequences

> Branch-miss rate = mispredicts / branches per stage and algorithm. Normalises out the per-stage call count so two algorithms running the same number of calls can be compared on prediction quality. A 5% miss rate on chaining is bad; a 1% miss rate on minimizer generation is normal.

#### `headline_largest_per_stage_branch_misses` — Per-stage branch mispredictions (every algorithm) — largest sequences

> Same shape as the cycles chart but Y is mean branch-mispredict counter per call. The bigger the bar, the more the stage is hurt by branch prediction noise. SIMD code with horizontal mask reductions (e.g. the slow legacy SIMD-mm) typically shows up as taller bars than its scalar peers — the production two-stack-nthash path mostly avoids this by using a uniform per-w rebuild instead of a data-dependent rescan.

...

### Other charts (uncategorised)

#### `seq_length_vs_throughput` — Throughput (Mbp/s) vs. effective query length

> Aligner throughput, in megabases-per-second, derived as effective_query_length / mean_latency. Throughput-normalised view of Chart 1: a flat horizontal trace means the algorithm's per-base cost is constant across length classes; a sloping trace means the per-base cost changes (e.g. cache effects on long reads, or super-linear chaining). Use this view when comparing pipelines whose absolute latency differs by orders of magnitude — the units cancel out the length dependence.

#### `speedup_vs_minimap2_reference_dp` — Paired speedup vs. minimap2 (DP chaining) per length bucket

> Per-bucket paired speedup ratio vs. the minimap2 (DP) baseline. For each (length_bucket, error_bucket) cell we compute the cell's mean of `mm2_DP_mean_ns / variant_mean_ns` (equivalently: per-pair speedup, averaged over pairs). Bars above 1 = bioseq faster than minimap2; bars below 1 = slower. Pairing makes the comparison robust to one slow outlier dominating either operand. The two minimap2 traces are excluded (DP would always be at 1.0 by construction; RMQ is a different reference).